# PathRAG (2025)
[[paper]](https://arxiv.org/abs/2502.14518)<br>
PathRAG = Path-level Retrieval-Augmented Generation

__PathRAG__ — это фреймворк для реализации Retrieval-Augmented Generation на основе графов знаний (Knowledge Graphs), который фокусируется на извлечении и фильтрации конкретных путей отношений между сущностями, а не просто семантически близких фрагментов текста или целых кластеров графа.

__Постановка задачи__<br>
Решается задача Multi-hop Question Answering и Complex Reasoning. Необходимо ответить на вопрос, требующий сопоставления фактов, которые могут быть распределены по разным документам или частям базы знаний, связанных между собой через цепочки логических связей.

__Мотивация__<br>
Классический Dense Retrieval (RAG) хорошо находит отдельные фрагменты текста, но плохо справляется с вопросами, где ответ требует прохода по "цепочке" фактов (например, "Как связан основатель компании X с технологией Y?"). Существующие графовые подходы часто страдают от избыточности: они либо извлекают слишком широкие окрестности узлов, либо целые сообщества (communities), что засоряет контекстное окно LLM нерелевантной информацией и увеличивает стоимость инференса.

__Существующие подходы__<br>
На момент появления PathRAG основными альтернативами были:
- Naive RAG (2020): поиск по векторному сходству фрагментов. Не видит структурных связей между документами.
- GraphRAG (Microsoft, 2024): строит иерархию сообществ в графе и генерирует суммаризации для каждого уровня. Основной минус — высокая вычислительная сложность индексации и избыточность токенов при генерации (Community Summaries часто содержат много лишнего).
- G-RAG / HippoRAG (2024): используют алгоритмы типа PageRank для поиска важных узлов. Минус — они фокусируются на важности отдельных узлов, а не на специфике отношений (ребрах) между ними в контексте конкретного запроса.

__Идея__<br>
Вместо того чтобы давать модели целые кластеры графа, нужно выделить только те пути (paths), которые соединяют ключевые сущности запроса. Новизна PathRAG заключается в двухэтапном процессе: сначала генерируется множество потенциальных путей в графе, а затем применяется Path Pruning (отсечение) — алгоритм ранжирования, который оставляет только те цепочки связей, которые семантически важны для ответа на конкретный вопрос.

__Архитектура__<br>
Система состоит из четырех ключевых компонентов:
1. Entity Extractor: модуль на базе LLM, который выделяет из запроса ключевые сущности (узлы графа).
2. Path Generator: алгоритм поиска путей (обычно на основе BFS или DFS) в графе знаний, соединяющий выделенные сущности.
3. Path Pruner: модель-ранжировщик (Cross-Encoder или легковесная LLM), которая оценивает релевантность каждого найденного пути.
4. Response Generator: финальная LLM, получающая на вход запрос и набор отфильтрованных путей в текстовом представлении.

__Алгоритм работы__<br>
Процесс инференса (поиска и генерации) выполняется по шагам:
1. Извлечение сущностей: Из запроса "Как разработки лаборатории X повлияли на стандарт Y?" извлекаются узлы "лаборатория X" и "стандарт Y".
2. Поиск подграфа: В глобальном графе знаний (построенном заранее из корпуса документов) ищутся все пути между этими узлами заданной длины (обычно до 3-4 прыжков).
3. Формирование кандидатов: Каждый путь представляется в виде текстовой последовательности троек: (Узел А — отношение1 — Узел Б — отношение2 — Узел В).
4. Pruning (Отсечение): Модель Path Pruner вычисляет Score для каждого пути. Пути с низким Score (шум или косвенные связи) удаляются. Это позволяет сократить объем контекста в 5-10 раз по сравнению с Community-based методами.
5. Генерация: Оставшиеся пути подаются в Prompt. LLM синтезирует ответ, опираясь на предоставленные логические цепочки.

__Обучение__<br>
PathRAG не всегда требует обучения основной LLM, но требует настройки Path Pruner:
1. Подготовка данных: Используются датасеты с графовой структурой (например, HotpotQA или MultiHop-RAG).
2. Контрастивное обучение: Pruner обучается отличать "золотые" пути (ведущие к правильному ответу) от случайных путей, существующих в графе, но не объясняющих связь в контексте вопроса.
3. Минимизируется Loss между предсказанной релевантностью пути и фактическим наличием в нем информации для ответа.

__Результаты__<br>
Авторы сравнивали метод на бенчмарках MultiHop-RAG и NarrativeQA:
- PathRAG превзошел Microsoft GraphRAG по метрике F1-score на 12-15% на сложных многоходовых вопросах.
- Эффективность использования контекста: благодаря Path Pruning, длина промпта сократилась на 70% по сравнению с методами извлечения сообществ, что привело к снижению задержки (Latency) генерации ответа в 2.5 раза.
- Точность извлечения фактов (Retrieval Recall) увеличилась на 18пп по сравнению с обычным Dense Retrieval, так как граф позволил найти связи, которые не имели высокого косинусного сходства в векторном пространстве.

## 📝 Критический анализ

```markdown
# PathRAG (2025)
---
[[paper]](https://arxiv.org/abs/2502.14518)<br>
PathRAG = Path-level Retrieval-Augmented Generation

__PathRAG__ — это фреймворк для Retrieval-Augmented Generation на основе графов знаний, фокусирующийся на извлечении и фильтрации путей отношений между сущностями.

__Постановка задачи__<br>
Решается задача Multi-hop Question Answering и Complex Reasoning, требующая сопоставления фактов, распределенных по разным документам или частям базы знаний.

__Мотивация__<br>
Классический Dense Retrieval плохо справляется с вопросами, требующими прохода по "цепочке" фактов. Графовые подходы часто извлекают избыточную информацию, увеличивая стоимость инференса.

__Существующие подходы__<br>
- Naive RAG (2020): поиск по векторному сходству, не видит структурных связей.
- GraphRAG (Microsoft, 2024): строит иерархию сообществ, но имеет высокую вычислительную сложность.
- G-RAG / HippoRAG (2024): фокусируются на важности узлов, а не на специфике отношений.

__Идея__<br>
Выделение только тех путей, которые соединяют ключевые сущности запроса. PathRAG использует двухэтапный процесс: генерация путей и Path Pruning для отбора семантически важных цепочек.

__Архитектура__<br>
Система состоит из:
1. Entity Extractor: выделяет ключевые сущности.
2. Path Generator: ищет пути в графе знаний.
3. Path Pruner: оценивает релевантность путей.
4. Response Generator: генерирует ответ на основе отфильтрованных путей.

<img src="img/img.png" width=500>

__Алгоритм работы__<br>
1. Извлечение сущностей из запроса.
2. Поиск путей в графе знаний.
3. Формирование кандидатов в виде текстовых последовательностей.
4. Pruning: удаление нерелевантных путей, сокращение объема контекста.
5. Генерация ответа с использованием оставшихся путей.

__Обучение__<br>
PathRAG требует настройки Path Pruner:
1. Подготовка данных с графовой структурой.
2. Контрастивное обучение для отличия "золотых" путей.
3. Минимизация Loss между предсказанной релевантностью и фактической информацией.

__Результаты__<br>
- PathRAG превзошел Microsoft GraphRAG по F1-score на 12-15% на сложных вопросах.
- Длина промпта сократилась на 70%, снижая задержку генерации в 2.5 раза.
- Точность извлечения фактов увеличилась на 18пп по сравнению с Dense Retrieval.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Импортируем необходимые библиотеки
import networkx as nx
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Entity Extractor: Извлечение ключевых сущностей из запроса
def extract_entities(question):
    # Используем предварительно обученную модель для извлечения сущностей
    nlp = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english")
    entities = nlp(question)
    return [entity['word'] for entity in entities]

# Пример запроса
question = "Как разработки лаборатории X повлияли на стандарт Y?"
entities = extract_entities(question)
print("Извлеченные сущности:", entities)

# 2. Path Generator: Поиск путей в графе знаний
def generate_paths(graph, entities, max_length=3):
    # Используем BFS для поиска путей между сущностями
    paths = []
    for start_entity in entities:
        for end_entity in entities:
            if start_entity != end_entity:
                for path in nx.all_simple_paths(graph, source=start_entity, target=end_entity, cutoff=max_length):
                    paths.append(path)
    return paths

# Создаем пример графа знаний
G = nx.Graph()
G.add_edges_from([
    ("лаборатория X", "разработка A"),
    ("разработка A", "технология B"),
    ("технология B", "стандарт Y"),
    ("лаборатория X", "исследование C"),
    ("исследование C", "стандарт Y")
])

# Генерируем пути
paths = generate_paths(G, entities)
print("Найденные пути:", paths)

# 3. Path Pruner: Отсечение нерелевантных путей
def prune_paths(paths, question):
    # Используем модель для оценки релевантности пути
    pruner = pipeline("text-classification", model="cross-encoder/nli-deberta-v3-base")
    pruned_paths = []
    for path in paths:
        path_text = " — ".join(path)
        score = pruner(f"{question} [SEP] {path_text}")[0]['score']
        if score > 0.5:  # Пороговое значение для отсечения
            pruned_paths.append(path)
    return pruned_paths

# Отсечение путей
pruned_paths = prune_paths(paths, question)
print("Отфильтрованные пути:", pruned_paths)

# 4. Response Generator: Генерация ответа на основе отфильтрованных путей
def generate_response(pruned_paths, question):
    # Используем LLM для генерации ответа
    response_generator = pipeline("text-generation", model="gpt2")
    context = " ".join([" — ".join(path) for path in pruned_paths])
    prompt = f"Вопрос: {question}\nКонтекст: {context}\nОтвет:"
    response = response_generator(prompt, max_length=100, num_return_sequences=1)
    return response[0]['generated_text']

# Генерация ответа
response = generate_response(pruned_paths, question)
print("Сгенерированный ответ:", response)
```

### Комментарии к коду:

1. **Entity Extractor**: Используется модель для извлечения именованных сущностей из запроса. Это позволяет выделить ключевые узлы графа, которые будут использоваться для поиска путей.

2. **Path Generator**: Используется алгоритм BFS для поиска всех возможных путей между извлеченными сущностями в графе знаний. Это позволяет найти все потенциальные цепочки, которые могут быть релевантны для ответа на вопрос.

3. **Path Pruner**: Используется модель для оценки релевантности каждого найденного пути. Путям присваивается оценка, и только те, которые превышают определенный порог, остаются для дальнейшего использования.

4. **Response Generator**: Используется LLM для генерации ответа на основе отфильтрованных путей. Это позволяет модели сосредоточиться на наиболее релевантной информации и сгенерировать более точный ответ.